# WeVisOne-VL 2B — detect, count, point, and cut out on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/WeVisOne-VL-2B-detect-segment.ipynb)

[`tencent/WeVisOne-VL-2B`](https://huggingface.co/tencent/WeVisOne-VL-2B) is Tencent WeChat Vision's 2.4B Qwen3-VL model (Apache 2.0). One checkpoint does **open-vocabulary boxes**, **referring / pointing**, **counting**, and **pixel-level masks / alpha mattes** ("find this and cut it out"). A 9.5M-parameter decoder (`mask_decoder.pt`) sits on the VLM; there is no SAM. The Hub weights are about 4.9 GB.

This is not another receipt-OCR or image-Q&A notebook. MiniCPM-V, LightOnOCR, and the AIBackends vision receipt notebook already cover read-the-page. Here the model has to **find, count, point at, and cut out** things you name.

**Licence:** Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The card's examples load `dtype="bfloat16"`. A T4 has no native bfloat16 and no FlashAttention 2, so this notebook loads **float16** and **SDPA** (eager if SDPA fails). Weights are about 4.9 GB; a 1024 px image sits around 5–8 GB. The 4B checkpoint is stronger on the card's tables and is a one-line swap, but it is roughly twice the weights and is not the T4 default here.

**Package:** inference is `wevisone_vl`, not a raw `transformers` generate loop. The official extra pins `torch==2.11.0`, which would replace Colab's CUDA build, so the install cell uses `--no-deps` and keeps Colab's torch.

**Links:** [2B card](https://huggingface.co/tencent/WeVisOne-VL-2B) · [4B card](https://huggingface.co/tencent/WeVisOne-VL-4B) · [GitHub / cookbooks](https://github.com/Tencent/WeVisOne-VL) · [paper](https://arxiv.org/abs/2610.31329) · [2B demo](https://huggingface.co/spaces/hugging-apps/wevisone-vl-2b)


In [ ]:
%pip install -q "transformers==5.12.1" "accelerate==1.14.0" "huggingface-hub>=0.34" "safetensors>=0.5" "supervision==0.31.0" matplotlib pillow
%pip install -q --no-deps "git+https://github.com/Tencent/WeVisOne-VL.git@9a6bcd85404505df0d05cfb5d97512ccacf35dee"

# transformers==5.12.1 is the wevisone-vl 1.0.0 pin (the checkpoint was saved with 5.5.3).
# --no-deps skips the package's torch==2.11.0 / torchvision==0.26.0 pins so Colab's CUDA torch stays.
# supervision draws the official box / point overlays. pillow and matplotlib are already on Colab.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


In [ ]:
import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.5.3"), (
    f"transformers {transformers.__version__} is loaded; WeVisOne-VL needs 5.5.3+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention 2. Keep PyTorch SDPA on the mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
# The card and package default to bfloat16. Force float16 on Turing (T4) and older.
DTYPE = "bfloat16" if major >= 8 else "float16"

print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


## 1. Load WeVisOne-VL 2B

`WeVisOneVL` from `wevisone_vl` loads the Qwen3-VL checkpoint **and** `mask_decoder.pt`. The first run downloads about 4.9 GB. `attention_implementation` defaults to `sdpa` in the package; FlashAttention 2 is not used.

`max_pixels` is capped at `1024 * 32 * 32` (the matting eval budget) so a free T4 keeps a margin. The package default is `2025 * 32 * 32` — raise it on an L4 or A100. Temperature stays at `0.0` (greedy), which is how the card and cookbooks run perception.


In [ ]:
from wevisone_vl import TASKS, WeVisOneVL

MODEL_ID = "tencent/WeVisOne-VL-2B"  # swap to tencent/WeVisOne-VL-4B on a larger GPU
MIN_PIXELS = 16 * 32 * 32
MAX_PIXELS = 1024 * 32 * 32  # T4-safe; package default is 2025 * 32 * 32


def load_wevisone(attn):
    return WeVisOneVL(
        MODEL_ID,
        backend="transformers",
        device="auto",
        dtype=DTYPE,
        attention_implementation=attn,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
    )


try:
    model = load_wevisone("sdpa")
    ATTN = "sdpa"
except Exception as exc:
    print("sdpa load failed, retrying eager:", exc)
    model = load_wevisone("eager")
    ATTN = "eager"

print("tasks:", ", ".join(TASKS))
print("attention:", ATTN, "| dtype:", model.dtype, "| device:", model.device)
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


## 2. Helpers and the shared dog photo

The photo is the official cookbook image (`dogs.jpg`). Overlays use `wevisone_vl.visualization`, which parses the model's `<ref>…</ref><box>…</box>` / `<point>…</point>` tokens (coordinates are 0–1000, not pixels). `result.boxes` / `result.masks` fill in only when the reply emits `<OBJ>` — that is the dense (mask / matte) path.


In [ ]:
import io
import time

import requests
from IPython.display import display
from PIL import Image

from wevisone_vl.visualization import (
    combined_alpha,
    make_cutout,
    parse_predictions,
    render_overlay,
)

DEMO_BASE = (
    "https://raw.githubusercontent.com/Tencent/WeVisOne-VL/"
    "9a6bcd85404505df0d05cfb5d97512ccacf35dee/demo/examples/images"
)
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}


def fetch_image(url):
    for attempt in range(4):
        response = requests.get(url, headers=UA, timeout=120)
        if response.status_code not in (429, 500, 502, 503, 504) or attempt == 3:
            response.raise_for_status()
            return Image.open(io.BytesIO(response.content)).convert("RGB")
        retry_after = response.headers.get("Retry-After", "")
        time.sleep(int(retry_after) if retry_after.isdigit() else 2 ** (attempt + 1))


def vram_line():
    print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


def show_boxes_and_points(image, result, title):
    preds = parse_predictions(result.response, image.width, image.height)
    n_box = sum(1 for items in preds.values() for item in items if item["type"] == "box")
    n_point = sum(1 for items in preds.values() for item in items if item["type"] == "point")
    print(title)
    print("Prompt:", result.prompt)
    print("Response:", result.response)
    print(f"Parsed labels: {list(preds)} | boxes: {n_box} | points: {n_point} | count: {n_box or n_point}")
    print(f"Latency: {result.latency_s:.2f}s")
    vram_line()
    display(render_overlay(image, result))
    return preds


dogs = fetch_image(f"{DEMO_BASE}/dogs.jpg")
print("dogs.jpg", dogs.size)
display(dogs)


## 3. Phrase → boxes + count

Two object-level calls on the same photo:

* `task="detection"` — open-vocabulary category list (not a closed COCO set)
* `task="count_box"` — a phrase, one box per instance, and a count

Coordinates in the reply are 0–1000. The overlay converts them to pixels.


In [ ]:
detect_result = model.predict(
    dogs,
    task="detection",
    query=["dog", "chair", "bush"],
    max_new_tokens=1024,
    temperature=0.0,
)
show_boxes_and_points(dogs, detect_result, "detection · dog / chair / bush")

count_result = model.predict(
    dogs,
    task="count_box",
    query="dogs",
    max_new_tokens=1024,
    temperature=0.0,
)
show_boxes_and_points(dogs, count_result, "count_box · dogs")


## 4. Grounding and pointing

`task="referring"` grounds a natural-language phrase to a box (or `None` if the thing is not there). `task="point"` returns a click-ready point per instance. That is the gap versus MiniCPM-V / LightOnOCR: those models describe or transcribe; this one has to **put a box or a point on the thing you named**.


In [ ]:
refer_result = model.predict(
    dogs,
    task="referring",
    query="the black and white dog",
    max_new_tokens=256,
    temperature=0.0,
)
show_boxes_and_points(dogs, refer_result, "referring · the black and white dog")

point_result = model.predict(
    dogs,
    task="point",
    query="dogs",
    max_new_tokens=256,
    temperature=0.0,
)
show_boxes_and_points(dogs, point_result, "point · dogs")

absent_result = model.predict(
    dogs,
    task="referring",
    query="a red fire hydrant",
    max_new_tokens=64,
    temperature=0.0,
)
print("absent target · a red fire hydrant")
print("Prompt:", absent_result.prompt)
print("Response:", absent_result.response)
print("(The card says a missing target should come back as None, not a hallucinated box.)")


## 5. Mask and alpha cutout — find and cut out anything

`task="matting"` is the card's "find this and cut it out" path: the VLM writes a box plus an `<OBJ>` token, then the 9.5M decoder emits a full-resolution alpha matte. `task="segmentation"` uses the same prompt family; the only difference in this notebook is that the overlay is thresholded at 0.5 for a binary mask.

The photo is the official cookbook matting image (`seabreeze.jpg`), query `"person"`.


In [ ]:
import matplotlib.pyplot as plt

sea = fetch_image(f"{DEMO_BASE}/seabreeze.jpg")
print("seabreeze.jpg", sea.size)
display(sea)

matte_result = model.predict(
    sea,
    task="matting",
    query="person",
    max_new_tokens=256,
    temperature=0.0,
)
print("Prompt:", matte_result.prompt)
print("Response:", matte_result.response)
print(f"Decoded mattes: {len(matte_result.masks)} | latency: {matte_result.latency_s:.2f}s")
vram_line()

if not matte_result.masks:
    print(
        "No <OBJ> / mask came back. The text boxes above still stand; "
        "pixel decode needs the wevisone_vl mask decoder and an <OBJ> token in the reply."
    )
else:
    overlay = render_overlay(sea, matte_result)
    binary = render_overlay(sea, matte_result, mask_threshold=0.5)
    alpha = combined_alpha(matte_result.masks, sea.width, sea.height)
    cutout = make_cutout(sea, alpha)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(overlay)
    axes[0].set_title("alpha overlay")
    axes[1].imshow(binary)
    axes[1].set_title("binary mask (0.5)")
    axes[2].imshow(cutout)
    axes[2].set_title("RGBA cutout")
    for axis in axes:
        axis.axis("off")
    fig.suptitle("find and cut out: person")
    fig.tight_layout()
    plt.show()


## Caveats

- **Package, not raw transformers.** `from wevisone_vl import WeVisOneVL` is required. A plain `AutoModelForImageTextToText.generate` call will not load `mask_decoder.pt` or emit decoded mattes.
- **Dtype.** The card and CLI default to `bfloat16`. On a T4 that is the wrong type: this notebook forces `float16` when `sm < 80`. Do not pass `dtype="bfloat16"` on Colab's free GPU.
- **Attention.** No FlashAttention-2 on a T4. The package default is already `sdpa`; this notebook falls back to `eager` if SDPA refuses the stack.
- **Install.** `pip install wevisone-vl` / a deps-on Git install pulls `torch==2.11.0`. Keep `--no-deps` on Colab.
- **Pixels.** `max_pixels` is lowered for 16 GB. Raise it if cutouts look coarse and you have VRAM left.
- **Single-turn images only.** The card rejects video grids in the pixel decoder. Fine alpha edges are sensitive to resolution, occlusion, and how you phrase the query.
- **Not for identifying people.** The card limits intended use to research in grounding, counting, pointing, OCR localization, referring segmentation, and matting.
